In [0]:
pip install kaggle

In [0]:
#Set Environment Variables so the Kaggle package knows who I am:
import os
os.environ['KAGGLE_USERNAME'] = "PLACEHOLDER"
os.environ['KAGGLE_KEY'] = 'PLACEHOLDER'

#Choosing the dataset to download.
#Go to kaggle dataset page and copy the text after 'Kaggle.com/datasets/'
dataset_slug = "PLACEHOLDER"
print("Successfully downloading data from Kaggle...")

In [0]:
%sh
# 1. Download the Dataset using the Kaggle CLI
# It downloads as a .zip archive into a temporary folder
kaggle datasets download -d "PLACEHOLDER" -p /tmp/kaggle_data/ --unzip

In [0]:
import os
# 1.looking inside the temp directory to see the exact folder kaggle created 
downloaded_items = os.listdir('/tmp/kaggle_data')
print(f"Item Found in download folder: {downloaded_items}")

# 2. Use /tmp directory which is accessible on the compute
# Instead of DBFS root (which is disabled), use the local temp directory
target_dir = "/tmp/ecommerce_raw/"
os.makedirs(target_dir, exist_ok=True)

# 3.Moving the file by using standard Python file operations
# Based on our logs, Kaggle extracted a folder named ecommerce_dataset
#Let's see what is inside it and move the csv safely.
import shutil
source_path = "/tmp/kaggle_data/ecommerce_dataset/"

print("\n Moving files to accessible storage...")
for file in os.listdir(source_path):
    if file.endswith('.csv'):
        #Copy from download location to persistent temp location
        shutil.copy(os.path.join(source_path, file), os.path.join(target_dir, file))
        print(f" Successfully copied: {file}")

# 4. List the target directory to confirm files are there
print("\n Current Files in temp storage: ")
for file in os.listdir(target_dir):
    print(os.path.join(target_dir, file))

In [0]:
%sql
-- Replace 'main' and 'default' with your catalog and schema names if different
CREATE CATALOG IF NOT EXISTS main;
USE CATALOG main;
CREATE SCHEMA IF NOT EXISTS ecommerce;
USE SCHEMA ecommerce;

-- Create a Volume to hold your raw data files
CREATE VOLUME IF NOT EXISTS raw_data;

In [0]:
import os
import shutil

# Local path where your files currently are
local_src = "/tmp/ecommerce_raw/"

# The standard path format for Unity Catalog Volumes is: /Volumes/<catalog>/<schema>/<volume_name>
volume_dst = "/Volumes/main/ecommerce/raw_data"

# Ensure the directory inside the volume exists
os.makedirs(volume_dst, exist_ok=True)

print("Moving files to Unity Catalog Volume...")
for filename in os.listdir(local_src):
    if filename.endswith('.csv'):
        src_path = os.path.join(local_src, filename)
        dst_path = os.path.join(volume_dst, filename)
        shutil.copy(src_path, dst_path)
        print(f"Staged in Volume: {filename}")

In [0]:
# Ingestion to the bronze layer(Raw Table)
# Read all the CSV files from local cluster storage
# Note: We append "file:" to the path so that spark knows its local temp Storage
orders_raw = spark.read.format("csv")\
                        .option("header", "true")\
                        .option("inferSchema", "true")\
                        .load("/Volumes/main/ecommerce/raw_data/orders.csv")

order_items_raw = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("/Volumes/main/ecommerce/raw_data/order_items.csv")

product_raw = spark.read.format("csv")\
                        .option("header", "true")\
                        .option("inferSchema", "true")\
                        .load("/Volumes/main/ecommerce/raw_data/products.csv")

user_raw = spark.read.format("csv")\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load("/Volumes/main/ecommerce/raw_data/users.csv")

# Write all the data as official Bronze Delta Tables
# Since DBFS root is disabled, writing as manage tables automatically saves them securely in Databricks metadata Storage
orders_raw.write.format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema", "true")\
                .saveAsTable("bronze_orders")

order_items_raw.write.format("delta")\
                    .mode("overwrite")\
                    .option("overwriteSchema", "true")\
                    .saveAsTable("bronze_order_items")

product_raw.write.format("delta")\
                 .mode("overwrite")\
                 .option("overwriteSchema", "true")\
                 .saveAsTable("bronze_product")
                 
user_raw.write.format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema", "true")\
                .saveAsTable("bronze_user")

print("Broze Stage Completed: All the data successfully ingested into the bronze layer tables!")
print(f"Ingested {orders_raw.count()} orders and {product_raw.count()} products.")

In [0]:
# Data Cleansing in the Silver Layer
from pyspark.sql.functions import col, current_date

# Read from our brand- new bronze tables
order_bronze = spark.read.table("bronze_orders")
order_items_bronze = spark.read.table("bronze_order_items")
product_bronze = spark.read.table("bronze_product")
user_bronze = spark.read.table("bronze_user")

# Silver Table 1: Orders, let's clean oders table: remove duplicate order IDs and ensure order_id isn't null
silver_orders_df = order_bronze\
                        .dropDuplicates(["order_id"])\
                        .filter(col("order_id").isNotNull())\
                        .withColumn("processed_at", current_date())

# Let's clean order_item table: remove duplicate order_item IDs and ensure order_item_id isn't null
silver_order_items_df = order_items_bronze\
                            .select(col("order_item_id").cast("string"),
                            col("order_id").cast("string"),
                            col("product_id").cast("string"),
                            col("user_id").cast("string"),
                            col("quantity").cast("int"),
                            col("item_price").cast("double"),
                            col("item_total").cast("double")               \
                            ).dropDuplicates(["order_item_id"])\
                            .filter(col("order_item_id").isNotNull())

#Clean Product table: Clean up price column if needed, drop duplicates
silver_products_df = product_bronze\
                          .dropDuplicates(["product_id"])\
                          .filter(col("product_id").isNotNull())

#Save to Silver Layer
silver_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver_orders")
silver_order_items_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_order_items")
silver_products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_products")

print("Silver Stage Completed: All the data successfully ingested into the silver layer tables!")

In [0]:
# Stage 3: Business Insights in the Gold Layer (Distributed Joins and Aggregations)
# Which Product categories are generatin the most revenue?
from pyspark.sql.functions import sum, count, round

#1 Read the clean data from Silver
orders = spark.read.table("silver_orders")
order_items = spark.read.table("silver_order_items")
products = spark.read.table("silver_products")

#2. Now we will join the tables together using Regular Inner Join
# We match up orders and order_items by using the common column(KEY): order_id
orders_with_items = orders.join(order_items, on = "order_id", how="inner")

# Now We match up orders_with_items and products by using the common column(KEY): product_id
joined_df = orders_with_items.join(products, on="product_id", how = "inner")

#3. Aggregate: Calculate total Revenue and Order Count category-wise.
gold_analytics_df = joined_df.groupBy("category")\
                            .agg(round(sum(col("price") * col("quantity")), 2). alias("total_revenue"), count("order_id").alias("total_orders"))\
                            .sort(col("total_revenue").desc())
                            # Highest revenue will come first

#4. Save to gold
gold_analytics_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_category_revenue")

print("Gold Stage Completed: Gold Analytical tables generated!")